In [ ]:
/kaggle/input/datasets/birdy654/deep-voice-deepfake-voice-recognition
/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset
/kaggle/input/datasets/rohan576/capstone-step3-weights

In [3]:
import subprocess, sys, os, json, shutil
import numpy as np
from pathlib import Path

# ============================================================
#  STEP 3 — Clean Cross-Dataset Eval: ASVspoof → WaveFake
#  Inputs needed:
#    1. capstone-step3-weights  (your uploaded best.pth files)
#    2. wavefake-vocoders-subset (the zip you made in Colab)
# ============================================================

# ── Paths ────────────────────────────────────────────────────
WORK_DIR   = Path("/kaggle/working/project4")
AASIST_DIR = WORK_DIR / "aasist"
RESULTS    = WORK_DIR / "step3_results"
RESULTS.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Weights (your capstone-step3-weights dataset)
WEIGHTS_BASE = Path("/kaggle/input/datasets/rohan576/capstone-step3-weights/step3_weights")
AASIST_CKPT  = WEIGHTS_BASE / "aasist_weights/best.pth"
RAWNET2_CKPT = WEIGHTS_BASE / "rawnet2_weights/best.pth"

# WaveFake vocoders (your wavefake-vocoders-subset dataset)
# Kaggle auto-extracts zips, so the folder will be inside the dataset
WAVEFAKE_INPUT = Path("/kaggle/input/datasets/rohan576/wavefake-vocoders-subset")

# LJSpeech bonafide (downloaded at runtime)
LJSPEECH_DIR = Path("/kaggle/working/ljspeech/wavs")

# ── 1. Install deps ───────────────────────────────────────────
subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "einops", "scikit-learn", "soundfile", "librosa"], check=True)

import torch, soundfile as sf, librosa
print(f"CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0)}")

# ── 2. Verify weight files ────────────────────────────────────
assert AASIST_CKPT.exists(),  f"❌ AASIST weights not found: {AASIST_CKPT}"
assert RAWNET2_CKPT.exists(), f"❌ RawNet2 weights not found: {RAWNET2_CKPT}"
print(f"✓ AASIST  weights: {AASIST_CKPT}  ({AASIST_CKPT.stat().st_size/1e6:.1f} MB)")
print(f"✓ RawNet2 weights: {RAWNET2_CKPT}  ({RAWNET2_CKPT.stat().st_size/1e6:.1f} MB)")

# ── 3. Locate WaveFake vocoders ───────────────────────────────
# Kaggle may auto-extract the zip or keep it — handle both cases
print("\n── Locating WaveFake vocoders...")
WAVEFAKE_OUT = Path("/kaggle/working/wavefake_extracted")

if not WAVEFAKE_OUT.exists() or not any(WAVEFAKE_OUT.iterdir()):
    # Check if Kaggle auto-extracted the zip
    auto_extracted = list(WAVEFAKE_INPUT.rglob("ljspeech_*"))
    if auto_extracted:
        # Files are already accessible in the input directory — use directly
        WAVEFAKE_OUT = WAVEFAKE_INPUT
        print(f"✓ WaveFake already extracted at {WAVEFAKE_INPUT}")
    else:
        # Kaggle kept the zip — extract it manually
        zip_files = list(WAVEFAKE_INPUT.rglob("*.zip"))
        assert zip_files, f"❌ No zip or extracted folders found in {WAVEFAKE_INPUT}"
        WAVEFAKE_OUT.mkdir(parents=True, exist_ok=True)
        print(f"── Extracting {zip_files[0].name}...")
        subprocess.run(["unzip", "-q", str(zip_files[0]),
                        "-d", str(WAVEFAKE_OUT)], check=True)
        print("✓ Extracted")

# Find vocoder directories (folders containing .wav files)
vocoder_dirs = {}
for vdir in sorted(WAVEFAKE_OUT.rglob("*")):
    if vdir.is_dir() and "ljspeech_" in vdir.name:
        wavs = list(vdir.glob("*.wav"))
        if wavs:
            vocoder_dirs[vdir.name] = wavs

assert vocoder_dirs, f"❌ No vocoder directories found. Check contents of {WAVEFAKE_OUT}"
print("\n── WaveFake vocoders found:")
for name, wavs in sorted(vocoder_dirs.items()):
    print(f"  {name}/  ({len(wavs)} files)")

# ── 4. Download LJSpeech (bonafide) ──────────────────────────
if not LJSPEECH_DIR.exists() or len(list(LJSPEECH_DIR.glob("*.wav"))) == 0:
    print("\n── Downloading LJSpeech (bonafide audio, ~2.6GB)...")
    lj_tar = Path("/kaggle/working/LJSpeech-1.1.tar.bz2")
    if not lj_tar.exists():
        subprocess.run([
            "wget", "-q", "--show-progress",
            "https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2",
            "-O", str(lj_tar)
        ], check=True)
    print("── Extracting LJSpeech...")
    subprocess.run(["tar", "-xjf", str(lj_tar), "-C", "/kaggle/working/"], check=True)
    lj_extracted = Path("/kaggle/working/LJSpeech-1.1")
    if lj_extracted.exists():
        lj_extracted.rename("/kaggle/working/ljspeech")
    lj_tar.unlink()
    print(f"✓ LJSpeech ready ({len(list(LJSPEECH_DIR.glob('*.wav')))} files)")
else:
    print(f"✓ LJSpeech already present ({len(list(LJSPEECH_DIR.glob('*.wav')))} files)")

bona_files = sorted(LJSPEECH_DIR.glob("*.wav"))
assert bona_files, f"❌ No LJSpeech wavs found in {LJSPEECH_DIR}"
print(f"✓ Bonafide: {len(bona_files)} files")

# ── 5. Clone AASIST (model architecture code only) ────────────
if not AASIST_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/clovaai/aasist.git",
                    str(AASIST_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", str(AASIST_DIR / "requirements.txt")], check=True)

for fpath in AASIST_DIR.rglob("*.py"):
    txt = fpath.read_text(encoding="utf-8")
    for old, new in [("np.float)", "float)"), ("np.float,", "float,"),
                     ("np.float ", "float "), ("num_workers=4", "num_workers=0"),
                     ("num_workers=8", "num_workers=0")]:
        txt = txt.replace(old, new)
    fpath.write_text(txt, encoding="utf-8")
print("✓ AASIST repo ready")

# ── 6. Load Models ────────────────────────────────────────────
sys.path.insert(0, str(AASIST_DIR))
from importlib import import_module

device = "cuda" if torch.cuda.is_available() else "cpu"

def load_model(architecture, first_conv, filts, extra_config, ckpt_path):
    model_config = {"architecture": architecture, "nb_samp": 64600,
                    "first_conv": first_conv, "filts": filts, **extra_config}
    module = import_module(f"models.{architecture}")
    model  = getattr(module, "Model")(model_config).to(device)
    state  = torch.load(ckpt_path, map_location=device, weights_only=False)
    if isinstance(state, dict) and "model" in state:
        state = state["model"]
    model.load_state_dict(state)
    model.eval()
    return model

aasist_model = load_model(
    architecture="AASIST", first_conv=128,
    filts=[70, [1, 32], [32, 32], [32, 64], [64, 64]],
    extra_config={"gat_dims": [64, 32], "pool_ratios": [0.5, 0.7, 0.5, 0.5],
                  "temperatures": [2.0, 2.0, 100.0, 100.0]},
    ckpt_path=AASIST_CKPT
)
print("✓ AASIST loaded")

rawnet2_model = load_model(
    architecture="RawNet2Spoof", first_conv=1024,
    filts=[20, [20, 20], [20, 128], [128, 128]],
    extra_config={"in_channels": 1, "blocks": [2, 4],
                  "nb_fc_node": 1024, "gru_node": 1024,
                  "nb_gru_layer": 3, "nb_classes": 2},
    ckpt_path=RAWNET2_CKPT
)
print("✓ RawNet2 loaded")

# ── 7. Audio Loader ───────────────────────────────────────────
TARGET_SR  = 16000
TARGET_LEN = 64600  # 4 sec @ 16kHz — matches AASIST training

def load_audio(filepath):
    wav, sr = sf.read(str(filepath), dtype="float32")
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != TARGET_SR:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=TARGET_SR)
    if len(wav) < TARGET_LEN:
        wav = np.pad(wav, (0, TARGET_LEN - len(wav)))
    else:
        wav = wav[:TARGET_LEN]
    return torch.FloatTensor(wav)  

# ── 8. Batch Scorer ───────────────────────────────────────────
def score_files(model, file_list, batch_size=32):
    scores = []
    for i in range(0, len(file_list), batch_size):
        batch_wavs = []
        for p in file_list[i:i+batch_size]:
            try:
                batch_wavs.append(load_audio(p))
            except Exception as e:
                print(f"  ⚠ Skipped {Path(p).name}: {e}")
                continue
        if not batch_wavs:
            continue
        batch_tensor = torch.stack(batch_wavs).to(device)
        with torch.no_grad():
            _, out       = model(batch_tensor)
            batch_scores = out[:, 1].cpu().numpy()
        scores.extend(batch_scores.tolist())
    return scores

# ── 9. Vectorized EER (memory-efficient via searchsorted) ────
def compute_eer(bona_scores, spoof_scores):
    bona  = np.array(bona_scores, dtype=np.float32)
    spoof = np.array(spoof_scores, dtype=np.float32)

    bona_sorted  = np.sort(bona)
    spoof_sorted = np.sort(spoof)

    thresholds = np.unique(np.concatenate([bona, spoof]))

    # FRR: fraction of bonafide scores below threshold (rejected)
    frr = np.searchsorted(bona_sorted,  thresholds, side="left")  / len(bona)
    # FAR: fraction of spoof scores at or above threshold (accepted as bona)
    far = 1.0 - np.searchsorted(spoof_sorted, thresholds, side="left") / len(spoof)

    idx = np.argmin(np.abs(far - frr))
    return float((far[idx] + frr[idx]) / 2 * 100)


# ── 10. Evaluate ─────────────────────────────────────────────
print("\n══════════════════════════════════════════════════")
print("  EVALUATION — Clean (no codec) Cross-Dataset")
print("  ASVspoof 2019 trained → WaveFake eval")
print("══════════════════════════════════════════════════\n")

results = {}

for model_name, model in [("AASIST", aasist_model), ("RawNet2", rawnet2_model)]:
    print(f"\n── Model: {model_name}")
    results[model_name] = {}

    print(f"  Scoring {len(bona_files)} bonafide (LJSpeech) files...")
    bona_scores = score_files(model, bona_files)

    for vocoder_name, fake_files in sorted(vocoder_dirs.items()):
        print(f"  Scoring {len(fake_files):5d} fake [{vocoder_name}]...", end="", flush=True)
        spoof_scores = score_files(model, fake_files)
        eer = compute_eer(bona_scores, spoof_scores)
        results[model_name][vocoder_name] = eer
        print(f"  EER = {eer:.2f}%")

# ── 11. Summary Table ─────────────────────────────────────────
print("\n\n══════════════════════════════════════════════════")
print("  RESULTS — EER (%) per Vocoder × Model")
print("  Clean audio — no codec (your baseline)")
print("══════════════════════════════════════════════════\n")

all_vocoders = sorted(set(v for m in results.values() for v in m.keys()))
header = f"{'Vocoder':<35} {'AASIST':>10} {'RawNet2':>10}"
print(header)
print("-" * len(header))
for voc in all_vocoders:
    a = results.get("AASIST",  {}).get(voc, "N/A")
    r = results.get("RawNet2", {}).get(voc, "N/A")
    print(f"  {voc:<33} "
          f"{f'{a:.2f}%' if isinstance(a, float) else a:>10} "
          f"{f'{r:.2f}%' if isinstance(r, float) else r:>10}")

print()
for model_name in ["AASIST", "RawNet2"]:
    vals = [v for v in results.get(model_name, {}).values() if isinstance(v, float)]
    if vals:
        print(f"  {model_name} Average EER: {np.mean(vals):.2f}%")

# ── 12. Save ─────────────────────────────────────────────────
results_path = RESULTS / "step3_clean_eval.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
shutil.copy(results_path, "/kaggle/working/step3_clean_eval.json")
print(f"\n✓ Results saved → /kaggle/working/step3_clean_eval.json")
print("\n► These are your CLEAN BASELINE EER values.")
print("► Compare against codec-degraded EER in Step 4.")


CUDA: True, GPU: Tesla T4
✓ AASIST  weights: /kaggle/input/datasets/rohan576/capstone-step3-weights/step3_weights/aasist_weights/best.pth  (1.3 MB)
✓ RawNet2 weights: /kaggle/input/datasets/rohan576/capstone-step3-weights/step3_weights/rawnet2_weights/best.pth  (70.5 MB)

── Locating WaveFake vocoders...
✓ WaveFake already extracted at /kaggle/input/datasets/rohan576/wavefake-vocoders-subset

── WaveFake vocoders found:
  ljspeech_full_band_melgan/  (2000 files)
  ljspeech_hifiGAN/  (2000 files)
  ljspeech_melgan/  (2000 files)
  ljspeech_melgan_large/  (2000 files)
  ljspeech_multi_band_melgan/  (2000 files)
  ljspeech_parallel_wavegan/  (2000 files)
  ljspeech_waveglow/  (2000 files)
✓ LJSpeech already present (13100 files)
✓ Bonafide: 13100 files
✓ AASIST repo ready
✓ AASIST loaded
✓ RawNet2 loaded

══════════════════════════════════════════════════
  EVALUATION — Clean (no codec) Cross-Dataset
  ASVspoof 2019 trained → WaveFake eval
═════════════════════════════════════════════════

KeyboardInterrupt: 